# 寶貝專用：雲端跑「去拒絕模型」（100% 不拒色）

這個筆記本用 Google 免費 GPU 跑 abliterated（去拒絕版）模型，
然後開一個公網接口，你的 Telegram 寶貝就能直接接上去用。

## 使用步驟（照順序點）

1. 上方選單：**執行階段 → 變更執行階段類型 → 硬體加速器選 T4 GPU → 儲存**
2. 從第一個程式碼格子開始，**點每個格子左邊的 ▶ 播放鈕**，等它跑完再點下一個
3. 跑到「開隧道」那格，畫面會印出一行網址：`https://xxxx.trycloudflare.com/v1`
4. **複製那個網址**，去 Telegram 對寶貝打：`/接colab https://xxxx.trycloudflare.com/v1`
5. 寶貝回「接上了」之後，你聊的每一句都走這個 100% 不拒色的模型

## 注意
- 免費 Colab 會閒置斷線（約幾十分鐘沒用），或跑滿 12 小時被踢。
- 斷了之後寶貝會自動回落千問，不用怕。
- 要再用就重開這個筆記本，重新跑一遍，拿到**新網址**再 `/接colab` 一次。
- 最後一格「心跳」要一直跑著，別停，能延長存活時間。


In [ ]:
# ① 安裝 Ollama（模型引擎）並啟動
!curl -fsSL https://ollama.com/install.sh | sh
import subprocess, time
subprocess.Popen(["ollama", "serve"])
time.sleep(5)
print("Ollama 啟動完成")


In [ ]:
# ② 下載去拒絕模型（約 5GB，等 3~10 分鐘，跑完才點下一格）
# 預設 8B：T4 跑得快又穩。想更聰明可換 huihui_ai/qwen3-abliterated:14b（較慢）
!ollama pull huihui_ai/qwen3-abliterated:8b


In [ ]:
# ③ 快速測試模型有沒有裝好（會印出模型名字就 OK）
!curl -s http://localhost:11434/api/tags


In [ ]:
# ④ 開免費公網隧道，跑完會印出「你的 API 網址」
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared
import subprocess, re
proc = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://localhost:11434"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
url = None
for _ in range(200):
    line = proc.stdout.readline()
    if not line:
        break
    print(line, end="")
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if m:
        url = m.group(0)
        break
if url:
    print("\n\n===== 你的 API 網址（去 Telegram 打 /接colab 這個）=====")
    print(url + "/v1")
else:
    print("沒抓到網址，重新跑這格看看")


In [ ]:
# ⑤ 心跳保活：這格要一直跑著，別停！每 60 秒戳一下自己，延長存活
import requests, time
while True:
    try:
        requests.get("http://localhost:11434/api/tags", timeout=5)
        print("💓 心跳 OK", flush=True)
    except Exception:
        print("💔 心跳失敗，重試", flush=True)
    time.sleep(60)


## 接到 Telegram 寶貝

拿到 `https://xxxx.trycloudflare.com/v1` 之後：

```
/接colab https://xxxx.trycloudflare.com/v1
```

寶貝會回「接上了」。之後聊天全部走這個去拒絕模型。

```
/断colab   → 手動斷開，回到千問
/模型      → 查看現在接了哪些模型
```

Colab 斷線（網頁關掉太久、12 小時上限）後，寶貝自動回落千問，不會壞。
